# The panel: expert consensus at the team level

The per-pick test (`08_backtest.ipynb`) compares fans to one grader. This
notebook compares them to the whole industry at once, using **René Bugner's
annual composite** (@RNBWCV): every year he compiles 18–29 outlets' instant
team grades — Kiper, PFF, CBS, NFL.com, USA Today, Sporting News, and more —
into a GPA per team. It is the best available operationalization of
"expert consensus," at the cost of resolution: grades exist per *team-class*,
not per pick, so this analysis runs on 32 teams × year.

**Provenance.** Chart images archived in `data/raw/bugner/` (sources:
Reddit-hosted charts for 2021/2025, Bugner's Bluesky for 2026, tweet
screenshots for 2022–2024). GPAs transcribed by vision into
`data/processed/consensus_team_grades.csv`; Cincinnati 2025 carries a noted
glyph ambiguity (2.06 adopted — the only reading consistent with the chart's
rank-32 sort position).

The fan counterpart is each fanbase's **mean pick sentiment across its class**
(picks with ≥10 comments). Two standardizations, because each answers a
different objection:

- **across teams within a year** — "which fanbases were happiest this April" —
  comparable to how the consensus GPA is used, but exposed to cross-fanbase
  baseline cheerfulness;
- **within each team across its six classes** — "was this class unusually well
  received *by this fanbase's own standards*" — immune to baseline differences,
  estimated from only six observations per team.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

PROCESSED = Path("..") / "data" / "processed"

consensus = pd.read_csv(PROCESSED / "consensus_team_grades.csv")
outcomes = pd.read_csv(PROCESSED / "draft_outcomes_2021_2025.csv")
sent = pd.read_csv(PROCESSED / "pick_sentiment.csv")

print(f"consensus: {len(consensus)} team-classes, "
      f"{consensus.groupby('season')['n_sources'].first().to_dict()} sources/year")

consensus: 192 team-classes, {2021: 18, 2022: 18, 2023: 29, 2024: 20, 2025: 24, 2026: 24} sources/year


### Team-class outcomes and signals

Outcome per team-class: the mean **above-slot residual** of its picks — how
much the class outperformed what its draft positions alone predicted, using
the same log-pick curve as the per-pick backtest (fitted on 2021–2024).

In [2]:
outcomes["log_pick"] = np.log(outcomes["pick"])
slot_fit = smf.ols("dr_av_pct ~ log_pick",
                   data=outcomes[outcomes.season <= 2024]).fit()
outcomes["resid"] = outcomes["dr_av_pct"] - slot_fit.predict(outcomes)

team_out = (outcomes.groupby(["season", "team"])
            .agg(class_resid=("resid", "mean"), n_picks=("resid", "size"))
            .reset_index())

fan = (sent[sent.n_comments >= 10]
       .groupby(["season", "team"])
       .agg(class_sent=("sent_mean", "mean"), n_scored=("sent_mean", "size"))
       .reset_index())

panel = (consensus
         .merge(team_out, on=["season", "team"], how="left")
         .merge(fan, on=["season", "team"], how="left"))

panel["gpa_z"] = (panel.groupby("season")["gpa"]
                  .transform(lambda s: (s - s.mean()) / s.std()))
panel["fan_year_z"] = (panel.groupby("season")["class_sent"]
                       .transform(lambda s: (s - s.mean()) / s.std()))
panel["fan_team_z"] = (panel.groupby("team")["class_sent"]
                       .transform(lambda s: (s - s.mean()) / s.std()))

train = panel[panel.season <= 2024].dropna(subset=["class_resid", "fan_year_z"])
hold = panel[panel.season == 2025].dropna(subset=["class_resid", "fan_year_z"])
print(f"train team-classes: {len(train)}   holdout 2025: {len(hold)}")

train team-classes: 128   holdout 2025: 32


### Do the experts' report cards predict anything?

Same question as the per-pick test, one level up: does the consensus GPA —
or the fanbase's class-level mood — predict which classes outperform their
draft slots? The outcome is already slot-adjusted, so a bare regression is
the honest test. HC3 errors throughout.

In [3]:
specs = {
    "consensus alone": "class_resid ~ gpa_z",
    "fans alone (year z)": "class_resid ~ fan_year_z",
    "fans alone (team z)": "class_resid ~ fan_team_z",
    "consensus + fans (year z)": "class_resid ~ gpa_z + fan_year_z",
    "consensus + fans (team z)": "class_resid ~ gpa_z + fan_team_z",
}
rows = []
for name, f in specs.items():
    fit = smf.ols(f, data=train).fit(cov_type="HC3")
    row = {"model": name, "n": int(fit.nobs)}
    for t in ["gpa_z", "fan_year_z", "fan_team_z"]:
        if t in fit.params:
            row[f"{t} coef"] = round(fit.params[t], 4)
            row[f"{t} p"] = round(fit.pvalues[t], 4)
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))

print("\nper-season correlation of consensus GPA with class outcome:")
for s, g in train.groupby("season"):
    print(f"  {s}: r = {g.gpa_z.corr(g.class_resid):+.3f}  (n={len(g)})")
print("\n2025 holdout: consensus r = "
      f"{hold.gpa_z.corr(hold.class_resid):+.3f}, "
      f"fans (year z) r = {hold.fan_year_z.corr(hold.class_resid):+.3f}")

                    model   n  gpa_z coef  gpa_z p  fan_year_z coef  fan_year_z p  fan_team_z coef  fan_team_z p
          consensus alone 128      0.0113   0.1265              NaN           NaN              NaN           NaN
      fans alone (year z) 128         NaN      NaN           0.0144        0.0510              NaN           NaN
      fans alone (team z) 128         NaN      NaN              NaN           NaN           0.0098        0.1946
consensus + fans (year z) 128      0.0106   0.1571           0.0138        0.0535              NaN           NaN
consensus + fans (team z) 128      0.0107   0.1483              NaN           NaN           0.0089        0.2217

per-season correlation of consensus GPA with class outcome:
  2021: r = -0.237  (n=32)
  2022: r = +0.356  (n=32)
  2023: r = +0.417  (n=32)
  2024: r = +0.011  (n=32)

2025 holdout: consensus r = -0.111, fans (year z) r = +0.047


### Do fans and the industry even agree?

Before asking who is *right*, it's worth asking whether they disagree at all —
if fan mood just mirrors the grade shows, the two signals are one signal.

In [4]:
print("correlation of fan class sentiment (year z) with consensus GPA (z):")
for s, g in panel.dropna(subset=["fan_year_z"]).groupby("season"):
    print(f"  {s}: r = {g.gpa_z.corr(g.fan_year_z):+.3f}")

correlation of fan class sentiment (year z) with consensus GPA (z):
  2021: r = +0.060
  2022: r = +0.053
  2023: r = -0.052
  2024: r = +0.154
  2025: r = +0.062
  2026: r = +0.353


### Receipts

The individual whiffs and hits that a correlation hides. `class_resid` is in
class-percentile points above slot expectation; ±0.10 is a big miss/hit.

In [5]:
t = train.copy()
t["gap"] = t.gpa_z - t.class_resid / t.class_resid.std()
show = ["season", "team", "gpa", "gpa_z", "class_sent", "class_resid"]

print("consensus darlings that flopped (top-8 graded, worst realized):")
darl = t[t.gpa_z > 0.8].nsmallest(8, "class_resid")
print(darl[show].round(3).to_string(index=False))

print("\nconsensus punching bags that hit (bottom-8 graded, best realized):")
bags = t[t.gpa_z < -0.8].nlargest(8, "class_resid")
print(bags[show].round(3).to_string(index=False))

consensus darlings that flopped (top-8 graded, worst realized):
 season team  gpa  gpa_z  class_sent  class_resid
   2024  ARI 3.61  1.085       0.232       -0.144
   2023  IND 3.77  1.594       0.207       -0.104
   2021  NWE 3.57  0.900       0.216       -0.099
   2021  MIA 3.64  1.017       0.173       -0.068
   2024  CHI 3.87  1.591       0.171       -0.035
   2024  KAN 3.68  1.222       0.230       -0.025
   2021  NYJ 3.69  1.100       0.214       -0.019
   2023  NYG 3.39  0.863       0.219       -0.014

consensus punching bags that hit (bottom-8 graded, best realized):
 season team  gpa  gpa_z  class_sent  class_resid
   2021  HOU 1.88 -1.911       0.175        0.205
   2022  LAR 2.16 -1.340       0.219        0.102
   2024  TEN 2.47 -1.133       0.175        0.091
   2024  DAL 2.64 -0.802       0.206        0.078
   2021  PIT 2.50 -0.880       0.209        0.067
   2023  DEN 2.42 -1.001       0.189        0.062
   2021  GNB 2.47 -0.930       0.254        0.059
   2024  DEN 2.55 

In [6]:
out = panel[["season", "team", "gpa", "n_sources", "gpa_z",
             "class_sent", "fan_year_z", "fan_team_z",
             "class_resid", "n_picks", "n_scored"]]
out_path = PROCESSED / "team_panel.csv"
out.to_csv(out_path, index=False)
print(f"wrote {len(out)} rows to {out_path}")

wrote 192 rows to ../data/processed/team_panel.csv
